# Análise de desempenho: Serial vs. Paralelo

Nesta etapa, será utilizado o arquivo **`comparacao_geral.csv`** para gerar as seguintes análises:

- tabela consolidada com os tempos de execução;
- tabela pivô por tamanho da entrada e método utilizado;
- gráfico de tempo de execução;
- gráfico de speedup;
- gráfico de eficiência.

In [4]:
%pip install plotly
import pandas as pd
import plotly.express as px
from pathlib import Path

candidatos = [
    Path("comparacao_serial_parallel/comparacao_geral.csv"),
    Path("serial_parallel_processing_refatorado/comparacao_serial_parallel/comparacao_geral.csv"),
    Path("/mnt/data/serial_parallel_processing_refatorado/comparacao_serial_parallel/comparacao_geral.csv"),
    Path("comparacao_geral.csv"),
]

csv_path = next((p for p in candidatos if p.exists()), None)

if csv_path is None:
    raise FileNotFoundError(
        "Não encontrei o arquivo 'comparacao_geral.csv'. "
        "Verifique se ele está dentro da pasta 'comparacao_serial_parallel'."
    )

print(f"Arquivo encontrado: {csv_path}")

df = pd.read_csv(csv_path)
df



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.9 MB 8.8 MB/s eta 0:00:02
   -------------------------------- ------- 8.1/9.9 MB 28.9 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 26.4 MB/s  0:00:00

   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwh

,metodo,n,m,p,tempo_segundos,speedup,eficiencia,processos,valido,observacao
0,Serial,10,20,10,0.000204,1.000000,1.000000,1,True,Baseline sequencial
1,Paralelo sem agrupamento,10,20,10,0.243951,0.000838,0.000209,4,True,Uma tarefa por célula (100 tarefas)
2,Paralelo por linha,10,20,10,0.210412,0.000971,0.000243,4,True,Uma tarefa por linha (10 tarefas)
3,Paralelo por blocos,10,20,10,0.214763,0.000952,0.000238,4,True,Blocos de 20 linhas
4,Serial,60,120,60,0.050253,1.000000,1.000000,1,True,Baseline sequencial
5,Paralelo sem agrupamento,60,120,60,0.276610,0.181673,0.045418,4,True,Uma tarefa por célula (3600 tarefas)
6,Paralelo por linha,60,120,60,0.301789,0.166515,0.041629,4,True,Uma tarefa por linha (60 tarefas)
7,Paralelo por blocos,60,120,60,0.229900,0.218584,0.054646,4,True,Blocos de 20 linhas
8,Serial,120,240,120,0.268687,1.000000,1.000000,1,True,Baseline sequencial
9,Paralelo sem agrupamento,120,240,120,0.486681,0.552080,0.138020,4,True,Uma tarefa por célula (14400 tarefas)


In [5]:

# Ajustes auxiliares para visualização
df["tamanho"] = df["n"].astype(str) + " x " + df["m"].astype(str)
ordem_metodos = [
    "Serial",
    "Paralelo sem agrupamento",
    "Paralelo por linha",
    "Paralelo por blocos",
]
df["metodo"] = pd.Categorical(df["metodo"], categories=ordem_metodos, ordered=True)
df = df.sort_values(["n", "metodo"]).reset_index(drop=True)

df


,metodo,n,m,p,tempo_segundos,speedup,eficiencia,processos,valido,observacao,tamanho
0,Serial,10,20,10,0.000204,1.000000,1.000000,1,True,Baseline sequencial,10 x 20
1,Paralelo sem agrupamento,10,20,10,0.243951,0.000838,0.000209,4,True,Uma tarefa por célula (100 tarefas),10 x 20
2,Paralelo por linha,10,20,10,0.210412,0.000971,0.000243,4,True,Uma tarefa por linha (10 tarefas),10 x 20
3,Paralelo por blocos,10,20,10,0.214763,0.000952,0.000238,4,True,Blocos de 20 linhas,10 x 20
4,Serial,60,120,60,0.050253,1.000000,1.000000,1,True,Baseline sequencial,60 x 120
5,Paralelo sem agrupamento,60,120,60,0.276610,0.181673,0.045418,4,True,Uma tarefa por célula (3600 tarefas),60 x 120
6,Paralelo por linha,60,120,60,0.301789,0.166515,0.041629,4,True,Uma tarefa por linha (60 tarefas),60 x 120
7,Paralelo por blocos,60,120,60,0.229900,0.218584,0.054646,4,True,Blocos de 20 linhas,60 x 120
8,Serial,120,240,120,0.268687,1.000000,1.000000,1,True,Baseline sequencial,120 x 240
9,Paralelo sem agrupamento,120,240,120,0.486681,0.552080,0.138020,4,True,Uma tarefa por célula (14400 tarefas),120 x 240


In [6]:

# Tabela pivô com os tempos lado a lado
tabela_tempos = (
    df.pivot(index="tamanho", columns="metodo", values="tempo_segundos")
      .reset_index()
)

tabela_tempos


metodo,tamanho,Serial,Paralelo sem agrupamento,Paralelo por linha,Paralelo por blocos
0,10 x 20,0.000204,0.243951,0.210412,0.214763
1,120 x 240,0.268687,0.486681,0.485644,0.334894
2,60 x 120,0.050253,0.276610,0.301789,0.229900


In [7]:

# Tabela resumida com speedup e eficiência apenas dos métodos paralelos
tabela_metricas = (
    df[df["metodo"] != "Serial"]
    [["tamanho", "metodo", "tempo_segundos", "speedup", "eficiencia", "processos", "valido", "observacao"]]
    .reset_index(drop=True)
)

tabela_metricas


,tamanho,metodo,tempo_segundos,speedup,eficiencia,processos,valido,observacao
0,10 x 20,Paralelo sem agrupamento,0.243951,0.000838,0.000209,4,True,Uma tarefa por célula (100 tarefas)
1,10 x 20,Paralelo por linha,0.210412,0.000971,0.000243,4,True,Uma tarefa por linha (10 tarefas)
2,10 x 20,Paralelo por blocos,0.214763,0.000952,0.000238,4,True,Blocos de 20 linhas
3,60 x 120,Paralelo sem agrupamento,0.276610,0.181673,0.045418,4,True,Uma tarefa por célula (3600 tarefas)
4,60 x 120,Paralelo por linha,0.301789,0.166515,0.041629,4,True,Uma tarefa por linha (60 tarefas)
5,60 x 120,Paralelo por blocos,0.229900,0.218584,0.054646,4,True,Blocos de 20 linhas
6,120 x 240,Paralelo sem agrupamento,0.486681,0.552080,0.138020,4,True,Uma tarefa por célula (14400 tarefas)
7,120 x 240,Paralelo por linha,0.485644,0.553260,0.138315,4,True,Uma tarefa por linha (120 tarefas)
8,120 x 240,Paralelo por blocos,0.334894,0.802305,0.200576,4,True,Blocos de 20 linhas


In [10]:
%pip install -U nbformat ipykernel plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "browser"

fig_tempo = px.line(
    df,
    x="tamanho",
    y="tempo_segundos",
    color="metodo",
    markers=True,
    title="Tempo de execução por tamanho e método"
)

fig_tempo.update_layout(
    xaxis_title="Tamanho da matriz",
    yaxis_title="Tempo de execução (s)"
)

fig_tempo.show()

In [14]:
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "browser"

fig_tempo = px.line(
    df,
    x="tamanho",
    y="tempo_segundos",
    color="metodo",
    markers=True,
    title="Tempo de execução por tamanho e método"
)

fig_tempo.update_layout(
    xaxis_title="Tamanho da matriz",
    yaxis_title="Tempo de execução (s)"
)

fig_tempo.show()

In [15]:

fig_eficiencia = px.bar(
    df_speedup,
    x="tamanho",
    y="eficiencia",
    color="metodo",
    barmode="group",
    text_auto=".2f",
    title="Eficiência dos métodos paralelos",
    labels={
        "tamanho": "Tamanho da matriz",
        "eficiencia": "Eficiência",
        "metodo": "Método"
    }
)

fig_eficiencia.update_layout(xaxis_title="Tamanho da matriz", yaxis_title="Eficiência")
fig_eficiencia.show()


In [16]:

# Melhor método por tamanho de entrada
melhores = (
    df.loc[df.groupby("tamanho")["tempo_segundos"].idxmin(), ["tamanho", "metodo", "tempo_segundos"]]
    .reset_index(drop=True)
    .rename(columns={"metodo": "melhor_metodo", "tempo_segundos": "melhor_tempo"})
)

melhores


,tamanho,melhor_metodo,melhor_tempo
0,10 x 20,Serial,0.000204
1,120 x 240,Serial,0.268687
2,60 x 120,Serial,0.050253


## Interpretação esperada

- **Serial** serve como baseline.
- **Paralelo sem agrupamento** tende a sofrer mais com overhead.
- **Paralelo por linha** reduz parte desse overhead.
- **Paralelo por blocos** tende a ser a estratégia mais equilibrada, pois diminui o custo de criação e coordenação das tarefas.

Use os gráficos e tabelas acima para colocar no slide:
1. uma tabela de tempos;
2. um gráfico de tempo;
3. um gráfico de speedup.
